# 04_4 · Cascada jerárquica: daño/seguro → categorías gruesas

Este cuaderno prueba una cascada de dos etapas y la compara con el mejor Transformer **plano** registrado por `04_2`. La referencia plana se escoge sólo por PR-AUC macro de validación; nunca por test. Ambos sistemas se evalúan sobre exactamente los mismos chunks de validación y test.

Por defecto, la puerta binaria aprovecha también chunks `SEGURO` no seleccionados en el 4:1, pero **únicamente si pertenecen a videos asignados a train**. Esta es una ablación de datos ampliados: puede mejorar la diversidad de negativos, aunque ya no aísla por sí sola el efecto de la arquitectura. Para que la clase daño no quede ahogada, la BCE binaria pondera positivos por el cociente completo `n_seguro/n_daño`. La segunda etapa conserva todos los daños de train y combina negativos difíciles de la puerta con seguros seleccionados por hash.

**Requisito:** ejecutar en `04_2` las evaluaciones de los Transformers y su comparación final, que crea `registro_modelos_comparables.json`.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'scripts_auxiliares' / 'experimentos_jerarquicos.py').exists():
            return candidate
    raise FileNotFoundError('No se encontró la raíz del proyecto.')

ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import Image, Markdown, display
from scripts_auxiliares import experimentos_jerarquicos as hj

print('Raíz:', ROOT)
print('PyTorch:', hj.torch.__version__)
print('Dispositivo:', hj._device())

## 1. Auditoría del experimento congelado

La carga falla de forma deliberada si cambió el SHA-256 del dataset, del manifiesto, de un checkpoint o de su evaluación. También comprueba que no existan chunks ni videos compartidos entre `train`, `validation` y `test`.

In [ ]:
context = hj.load_frozen_context()
audit = hj.context_summary(context)
display(audit)
print('Referencia plana:', audit.attrs['flat_reference'])
print('PR-AUC macro de validación usada para seleccionarla:',
      f"{audit.attrs['flat_selection_metric']:.4f}")
print('SHA-256 dataset:', context['dataset_sha256'])
print('SHA-256 manifiesto:', context['manifest_sha256'])

## 2. Metodología

La etapa 1 estima `p(cualquier daño)`. La etapa 2 estima cinco probabilidades multietiqueta y el puntaje final es `p(daño) × p(categoría | puerta)`. Esta versión blanda conserva el orden de riesgo para PR-AUC y evita que un único corte binario destruya toda la información. Para operación selectiva se calibran **sólo con validación** dos cortes: debajo del primero se permite auto-paso seguro con objetivo de recall de daño 0,97; por encima del segundo se exige precisión de daño 0,90. La zona intermedia se abstiene y solicita revisión.

La referencia y la cascada comparten encoder base y revisión exacta. Época y umbrales se eligen con validación. La comparación final usa PR-AUC macro, F1 macro, recall por categoría y falsos negativos. Los IC 95 % provienen de 1.000 remuestreos pareados de **videos completos**, preservando la dependencia de chunks del mismo video. La mejora sólo se acepta si todo el IC de Δ PR-AUC es positivo y el límite superior de Δ tasa de falsos negativos no supera cero.

Sólo se entrenan las cinco categorías gruesas. Las etiquetas finas y los flags transversales permanecen para trazabilidad/criterios, pero no se usan como objetivos ni predictores.

In [ ]:
# Configuración declarada antes de abrir el test.
EXPANDED_SAFE_GATE = True   # activado por defecto: sólo SEGURO de videos de train
FORCE = False               # True vuelve a entrenar y reemplaza esta variante
BOOTSTRAP_REPLICATES = 1_000

if EXPANDED_SAFE_GATE:
    expanded_train, expanded_info = hj.expanded_safe_gate_training_frame(context)
    display(expanded_info)
else:
    print('Control estricto 4:1: mismos rows de train que el modelo plano.')

## 3. Entrenamiento con barras de avance

La celda guarda el mejor checkpoint de cada etapa, historiales por época, probabilidades de los tres splits, umbrales, métricas, tablas, gráficos e informe Markdown. Si el resultado compatible ya existe y `FORCE=False`, sólo lo carga.

In [ ]:
result = hj.run_cascade_experiment(
    force=FORCE,
    bootstrap_replicates=BOOTSTRAP_REPLICATES,
    expanded_safe_gate=EXPANDED_SAFE_GATE,
)
print('Experimento:', result['experiment_label'])
print('Mejor época puerta:', result['training']['gate']['best_epoch'])
print('Mejor época categorías:',
      result['training']['conditional_categories']['best_epoch'])
print('Resultado:', hj.result_path(result['experiment_key']))
print('Informe:', hj.report_path(result['experiment_key']))

## 4. Comparación con el modelo plano

In [ ]:
tables = hj.load_experiment_tables(result['experiment_key'])
display(tables['comparison'].round(4))
display(tables['categories'].round(4))
display(tables['bootstrap'].round(4))

In [ ]:
figure_dir = hj.FIGURES_ROOT / result['experiment_key']
for name in ('comparacion_global_test.png',
             'recall_por_categoria_test.png',
             'bootstrap_deltas_test.png'):
    display(Image(filename=str(figure_dir / name)))

## 5. Lectura operativa y conclusión

In [ ]:
op = result['selective_operation']
display({
    'umbral_auto_seguro': op['low_auto_safe_threshold'],
    'umbral_auto_daño': op['high_auto_damage_threshold'],
    'tasa_revision_test': op['review_rate'],
    'cobertura_automatica_test': op['automatic_coverage'],
    'daños_auto_pasados_como_seguro':
        op['damage_automatic_safe_false_negatives'],
    'decision_estadistica': result['decision']['status'],
    'reemplazar_plano': result['decision']['replace_flat_model'],
})
display(Markdown(hj.report_path(result['experiment_key']).read_text(encoding='utf-8')))

## 6. Control opcional para aislar arquitectura

La ejecución siguiente está desactivada. Si se activa, entrena la misma cascada usando sólo el `train` 4:1 del modelo plano y guarda sus artefactos con otra clave. Comparar ambas variantes permite separar el efecto de la cascada del efecto de los seguros adicionales.

In [ ]:
RUN_MATCHED_4_TO_1_CONTROL = False
if RUN_MATCHED_4_TO_1_CONTROL:
    matched_result = hj.run_cascade_experiment(
        force=False,
        bootstrap_replicates=BOOTSTRAP_REPLICATES,
        expanded_safe_gate=False,
    )
    display(hj.load_experiment_tables(matched_result['experiment_key'])['comparison'])

## Referencias (APA 7)

Cawley, G. C., & Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. *Journal of Machine Learning Research, 11*, 2079–2107. https://www.jmlr.org/papers/v11/cawley10a.html

Efron, B., & Tibshirani, R. J. (1993). *An introduction to the bootstrap*. Chapman & Hall/CRC.

Geifman, Y., & El-Yaniv, R. (2017). Selective classification for deep neural networks. *International Conference on Learning Representations*. https://openreview.net/forum?id=ryMhqj0ct7

Naik, A., & Rangwala, H. (2017). Filter based taxonomy modification for improving hierarchical classification. *arXiv*. https://doi.org/10.48550/arXiv.1706.01214

Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLOS ONE, 10*(3), e0118432. https://doi.org/10.1371/journal.pone.0118432